## Cell 1 — GPU check + installs

Check if a GPU is available and install pinned `transformers` / `accelerate` plus `datasets` and `scikit-learn`.

In [ ]:
# Check GPU and install dependencies
import subprocess
import sys

import torch

gpu_ok = torch.cuda.is_available()
print(f"GPU available: {gpu_ok}")
if gpu_ok:
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected; training will run on CPU (slow).")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==4.40.0",
        "accelerate==0.29.0",
        "datasets",
        "scikit-learn",
    ]
)
print("Dependencies installed.")

## Cell 2 — Upload training data

Upload `training-data.json` (from `apps/server/src/scripts/` after running the cleaner).

In [ ]:
from collections import Counter
import json

from google.colab import files

uploaded = files.upload()
fname = next(iter(uploaded.keys()))
raw_bytes = uploaded[fname]
if isinstance(raw_bytes, str):
    raw_text = raw_bytes
else:
    raw_text = raw_bytes.decode("utf-8")

data = json.loads(raw_text)
if not isinstance(data, list):
    raise ValueError("Expected a JSON array of {text, label} objects")

print(f"File: {fname}")
print(f"Row count: {len(data)}")
labels = [row["label"] for row in data]
dist = Counter(labels)
print("Label distribution:")
for lab, n in sorted(dist.items(), key=lambda x: (-x[1], x[0])):
    print(f"  {lab}: {n}")

## Cell 3 — Config (edit before running)

Adjust hyperparameters and paths here.

In [ ]:
BASE_MODEL = "distilbert-base-uncased"  # or path if uploading existing checkpoint
OUTPUT_DIR = "/content/job-parser-model-new"
LABEL_ORDER = [
    "responsibility",
    "requirement",
    "experience",
    "benefit",
    "contact",
    "position",
    "other",
]
MAX_LEN = 128
BATCH_TRAIN = 16
BATCH_EVAL = 32
EPOCHS = 5
LR = 2e-5
SEED = 42

## Cell 4 — Prepare dataset

Stratified 85/15 split, tokenize with padding and truncation.

In [ ]:
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, set_seed

set_seed(SEED)
np.random.seed(SEED)

label2id = {lab: i for i, lab in enumerate(LABEL_ORDER)}
id2label = {i: lab for lab, i in label2id.items()}

unknown = {row["label"] for row in data if row["label"] not in label2id}
if unknown:
    raise ValueError(f"Labels not in LABEL_ORDER: {unknown}")

texts = [row["text"] for row in data]
labels = [label2id[row["label"]] for row in data]

train_texts, eval_texts, train_labels, eval_labels = train_test_split(
    texts,
    labels,
    test_size=0.15,
    stratify=labels,
    random_state=SEED,
)

print(f"Train size: {len(train_texts)} | Eval size: {len(eval_texts)}")


def split_dist(name, y):
    c = Counter(y)
    print(f"\n{name} class counts (by id):")
    for i in range(len(LABEL_ORDER)):
        print(f"  {i} {LABEL_ORDER[i]}: {c.get(i, 0)}")


split_dist("Train", train_labels)
split_dist("Eval", eval_labels)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def tokenize_batch(examples):
    enc = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )
    enc["labels"] = examples["labels"]
    return enc


train_ds = Dataset.from_dict({"text": train_texts, "labels": train_labels})
eval_ds = Dataset.from_dict({"text": eval_texts, "labels": eval_labels})

train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)
eval_tok = eval_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

print("\nTokenized columns:", train_tok.column_names)

## Cell 5 — Load model

`AutoModelForSequenceClassification` with 7 labels and label maps.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABEL_ORDER),
    id2label=id2label,
    label2id=label2id,
)

param_count = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {param_count:,}")
print(f"Trainable parameters: {trainable:,}")

## Cell 6 — Train

`Trainer` with eval/save each epoch, best checkpoint by `eval_loss`, FP16 on GPU, warmup 10%, weight decay 0.01.

In [ ]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}_checkpoints",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    logging_steps=max(1, len(train_tok) // (BATCH_TRAIN * 10)),
    report_to="none",
    save_total_limit=2,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

## Cell 7 — Evaluate

Per-class metrics and confusion matrix on the eval split.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred_out = trainer.predict(eval_tok)
y_pred = np.argmax(pred_out.predictions, axis=-1)
y_true = np.array(eval_labels)

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(len(LABEL_ORDER))),
        target_names=LABEL_ORDER,
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(LABEL_ORDER))),
)

print("\nConfusion matrix (rows = true label, columns = predicted label)")
col_w = max(14, max(len(h) for h in LABEL_ORDER) + 2)
header = "".join(f"{name[:col_w]:>{col_w}}" for name in LABEL_ORDER)
print(f"{'':<22}{header}")
print("-" * (22 + col_w * len(LABEL_ORDER)))
for i, tname in enumerate(LABEL_ORDER):
    row_cells = "".join(f"{cm[i, j]:>{col_w}}" for j in range(len(LABEL_ORDER)))
    print(f"{tname[:20]:<22}{row_cells}")

## Cell 8 — Save + download

Save model, tokenizer, and `label_map.json`, then zip and download `job-parser-model-new.zip`.

In [ ]:
import json
import os
import shutil

from google.colab import files

os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

label_map = {
    "id2label": {str(k): v for k, v in id2label.items()},
    "label2id": label2id,
}
with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump(label_map, f, indent=2)

parent = os.path.dirname(os.path.abspath(OUTPUT_DIR))
base = os.path.basename(OUTPUT_DIR.rstrip("/"))
zip_stem = os.path.join("/tmp", "job-parser-model-new")
zip_path = shutil.make_archive(zip_stem, "zip", root_dir=parent, base_dir=base)

size_bytes = os.path.getsize(zip_path)
print(f"Zip file: {zip_path}")
print(f"Zip size: {size_bytes:,} bytes ({size_bytes / (1024 * 1024):.2f} MB)")

files.download(zip_path)